# Notebook 04 — Modeling
## Huấn luyện 10 thuật toán ML + Cross-Validation + Hyperparameter Tuning

**Thuật toán:** Logistic Regression, Decision Tree, Random Forest, Extra Trees, SVM, KNN, Gradient Boosting, AdaBoost, XGBoost, LightGBM  
**Validation:** StratifiedKFold k=5, Metric chính: F1-macro & Recall(Dropout)


In [ ]:
import sys
sys.path.insert(0, '..')

import os
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from src.models.train import compare_models, train_best_model, save_comparison, MODELS
from src.models.evaluate import plot_model_comparison
from src.data.load_data import FEATURE_COLS

plt.rcParams['figure.dpi'] = 120
print(f'Import thành công ✅ — {len(MODELS)} mô hình: {list(MODELS.keys())}')

## 1. Load processed data

In [ ]:
X_train = pd.read_csv('../data/processed/X_train.csv').values
X_test  = pd.read_csv('../data/processed/X_test.csv').values
y_train = pd.read_csv('../data/processed/y_train.csv').squeeze().values.astype(int)
y_test  = pd.read_csv('../data/processed/y_test.csv').squeeze().values.astype(int)

print(f'X_train: {X_train.shape}, y_train: {y_train.shape}')
print(f'X_test : {X_test.shape},  y_test : {y_test.shape}')
print(f'Train class dist: {np.bincount(y_train)}')
print(f'Test  class dist: {np.bincount(y_test)}')

## 2. So sánh 10 thuật toán với Cross-Validation

In [ ]:
print(f'Training {len(MODELS)} thuật toán với StratifiedKFold (k=5)...')
print('Đây có thể mất 10-20 phút, vui lòng chờ...\n')

df_results = compare_models(X_train, y_train, cv=5)

print('\n=== KẾT QUẢ SO SÁNH ===')
df_results

In [ ]:
# Lưu bảng so sánh
save_comparison(df_results, path='../models/model_comparison.csv')

# Vẽ biểu đồ so sánh
fig = plot_model_comparison(
    df_results, metric='f1_macro',
    save_path='../reports/figures/04_model_comparison_f1.png'
)
plt.show()

fig2 = plot_model_comparison(
    df_results, metric='recall_macro',
    save_path='../reports/figures/04_model_comparison_recall.png'
)
plt.show()

## 3. Train & lưu từng mô hình riêng (.pkl)

In [ ]:
# Train tất cả mô hình trên toàn bộ X_train và lưu file .pkl riêng
# Tên file: tên mô hình chữ thường, khoảng trắng → '_'

os.makedirs('../models', exist_ok=True)

for model_name, model in MODELS.items():
    print(f'Training {model_name}...', end=' ')
    model.fit(X_train, y_train)
    filename = model_name.lower().replace(' ', '_') + '.pkl'
    save_path = f'../models/{filename}'
    joblib.dump(model, save_path)
    print(f'saved → {filename} ✅')

print('\nDanh sách file đã lưu:')
for f in sorted(os.listdir('../models')):
    if f.endswith('.pkl'):
        size_kb = os.path.getsize(f'../models/{f}') / 1024
        print(f'  {f:40s} {size_kb:8.1f} KB')

## 4. Hyperparameter Tuning — Mô hình tốt nhất (best_model.pkl)

In [ ]:
# Tự động chọn mô hình top 1 theo F1-macro để tuning
BEST_MODEL_NAME = df_results.iloc[0]['Model']
print(f'Mô hình được chọn để tuning: {BEST_MODEL_NAME}')

print('\nBắt đầu GridSearchCV...')
best_model = train_best_model(
    X_train, y_train,
    model_name=BEST_MODEL_NAME,
    tune=True,
    save_path='../models/best_model.pkl'
)

print('\nChecklist file:')
for f in ['../models/best_model.pkl', '../models/preprocessing_pipeline.pkl']:
    print(f'  {"✅" if os.path.exists(f) else "❌"} {f}')

## 5. Kiểm tra nhanh tất cả mô hình trên test set

In [ ]:
from sklearn.metrics import f1_score, accuracy_score

print(f'{"Mô hình":30s}  {"Accuracy":>10}  {"F1-macro":>10}')
print('-' * 55)
for model_name in MODELS.keys():
    filename = model_name.lower().replace(' ', '_') + '.pkl'
    path = f'../models/{filename}'
    if os.path.exists(path):
        m = joblib.load(path)
        y_pred = m.predict(X_test)
        acc = accuracy_score(y_test, y_pred)
        f1  = f1_score(y_test, y_pred, average='macro')
        print(f'{model_name:30s}  {acc:10.4f}  {f1:10.4f}')